# Part1

# Task1

In [4]:
from langchain_community.llms import Ollama

llm = Ollama(model="gemma3")

response = llm.invoke("Explain RAG in one sentence.")
print(response)

Retrieval-Augmented Generation (RAG) combines a large language model with an external knowledge source to generate more accurate and contextually relevant responses by first retrieving relevant information and then using that information to guide its generation.


# Part2

# Task2

In [6]:
!pip install wikipedia

  Using cached wikipedia-1.4.0-py3-none-any.whl


In [8]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k_results=3, lang="en")

docs = retriever.invoke("Large Language Models")

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:500])



--- Document 1 ---
A large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of text, designed for natural language processing tasks, especially language generation. The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots. LLMs can be fine-tuned for specific tasks or guided by prompt engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inhere

--- Document 2 ---
A large language model (LLM) is a type of machine learning model designed for natural language processing tasks such as language generation. LLMs are language models with many parameters, and are trained with self-supervised learning on a vast amount of text.


== List ==
For the training cost column, 1 petaFLOP-day = 1 petaFLOP/sec × 1 day = 8.64E19 FLOP. Also, only the largest model's cost is written.


== See also ==
List of chatbots
List of language m

# Task3

In [9]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="Retrieval Augmented Generation", load_max_docs=5)
docs = loader.load()


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Windows\Temp\ipykernel_20788\3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 580.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()


In [14]:
query = "What is RAG?"
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content[:300])


The term RAG was first introduced in a 2020 research paper.
to Ars Technica, "RAG is a way of improving LLM performance, in essence by blending the LLM process with a web search or other document look-up process to help LLMs stick to the facts." This method helps reduce AI hallucinations, which have caused chatbots to describe policies that don't exist, or
== RAG and LLM limitations ==
=== RAG key stages ===


# Part3

# Task4

In [16]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "lambda_mult": 0.5}
)

mmr_docs = mmr_retriever.invoke("RAG architecture")


# Task5

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

docs = multi_retriever.get_relevant_documents(
    "How does RAG improve LLM answers?"
)

for doc in docs:
    print(doc.page_content[:250])


# Task6

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)

compressed_docs = compression_retriever.get_relevant_documents(
    "Explain RAG"
)


# Part4

# Task7

In [2]:
!pip install pytube

In [5]:
pip install --upgrade pytube


Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install --upgrade youtube-transcript-api


Note: you may need to restart the kernel to use updated packages.


In [12]:
from youtube_transcript_api import YouTubeTranscriptApi

# Video ID
video_id = "cdiD-9MMpb0"

# Fetch transcript
transcript = YouTubeTranscriptApi.list_transcripts(video_id).find_transcript(['en']).fetch()

# Print
for t in transcript:
    print(f"{t['start']:.2f}s - {t['text']}")


AttributeError: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'

# Task8

In [ ]:
chunks = splitter.split_documents(docs)

yt_vectorstore = FAISS.from_documents(chunks, embeddings)
yt_retriever = yt_vectorstore.as_retriever()


# Task9

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=yt_retriever,
    return_source_documents=True
)

query = "What is the main topic of the video?"
result = qa_chain.invoke(query)

print(result["result"])


# Task10

In [ ]:
questions = [
    "What problem does the video solve?",
    "What tools are mentioned?",
    "Explain the architecture discussed",
    "What is RAG?",
    "What are the limitations?"
]

for q in questions:
    res = qa_chain.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {res['result']}")


In [ ]:
unknown_q = "What is the capital of Mars?"
res = qa_chain.invoke(unknown_q)

if not res["source_documents"]:
    print("Sorry, the answer is not available in the video content.")


# Part5

# Task11

Retriever-Based RAG vs Normal Prompting

Normal prompting: LLM answers from training data only

RAG: LLM uses external documents → factual & updated answers

- Why Vector Stores Are Critical

Enable semantic search

Scale to millions of documents

Faster & more accurate retrieval

- When to Use MMR vs Similarity Search

Similarity: focused, precise queries

MMR: broad topics, avoid redundancy

- Benefits of Multi-Query Retrieval

Handles vague questions

Improves recall

Captures different phrasings

- Importance of Contextual Compression

Removes irrelevant text

Saves tokens

Improves answer grounding